In [9]:
# Combine into one CAGR table
cagr_table = first_year_revenue.merge(last_year_revenue, on="Company ")
cagr_table["years"] = cagr_table["last_year"] - cagr_table["first_year"]
cagr_table["cagr_pct"] = (
    (cagr_table["last_revenue"] / cagr_table["first_revenue"]) ** (1 / cagr_table["years"]) - 1
) * 100

# Sort by CAGR descending
cagr_table = cagr_table.sort_values("cagr_pct", ascending=False)

# Show the result
print(cagr_table[["Company ", "first_year", "last_year", "years", "cagr_pct"]].round(1))

   Company   first_year  last_year  years  cagr_pct
11     AMZN        2009       2022     13      26.4
2      GOOG        2009       2022     13      21.0
0      AAPL        2009       2022     13      18.6
9      NVDA        2009       2022     13      17.2
3      PYPL        2014       2022      8      16.7
1      MSFT        2009       2022     13       9.9
10     INTC        2009       2022     13       4.6
5       PCG        2009       2022     13       3.8
7       MCD        2009       2022     13       0.1
4       AIG        2009       2022     13      -2.2
8       BCS        2009       2022     13      -3.0
6     SHLDQ        2009       2018      9     -10.8


In [8]:
# For each company, get the revenue at first_year and last_year
# This requires a merge — your first one in pandas

# Strategy: merge year_bounds back to df to get revenues at those years
first_year_revenue = df.merge(
    year_bounds[["first_year"]],
    left_on=["Company ", "Year"],
    right_on=["Company ", "first_year"],
    how="inner"
)[["Company ", "Year", "Revenue"]].rename(columns={"Year": "first_year", "Revenue": "first_revenue"})

last_year_revenue = df.merge(
    year_bounds[["last_year"]],
    left_on=["Company ", "Year"],
    right_on=["Company ", "last_year"],
    how="inner"
)[["Company ", "Year", "Revenue"]].rename(columns={"Year": "last_year", "Revenue": "last_revenue"})

print(first_year_revenue)
print()
print(last_year_revenue)

   Company   first_year  first_revenue
0      AAPL        2009      42905.000
1      MSFT        2009      58437.000
2      GOOG        2009      23651.000
3      PYPL        2014       8025.000
4       AIG        2009      75447.000
5       PCG        2009      13399.000
6     SHLDQ        2009      46770.000
7       MCD        2009      22744.700
8       BCS        2009      45992.040
9      NVDA        2009       3424.859
10     INTC        2009      35127.000
11     AMZN        2009      24509.000

   Company   last_year  last_revenue
0      AAPL       2022     394328.00
1      MSFT       2022     198270.00
2      GOOG       2022     282836.00
3      PYPL       2022      27518.00
4       AIG       2022      56437.00
5       PCG       2022      21680.00
6     SHLDQ       2018      16702.00
7       MCD       2022      23182.60
8       BCS       2022      30868.08
9      NVDA       2022      26914.00
10     INTC       2022      63054.00
11     AMZN       2022     513983.00


In [6]:
# Get first and last year per company
year_bounds = df.groupby("Company ").agg(
    first_year=("Year", "min"),
    last_year=("Year", "max"),
)
print(year_bounds)
print()

          first_year  last_year
Company                        
AAPL            2009       2022
AIG             2009       2022
AMZN            2009       2022
BCS             2009       2022
GOOG            2009       2022
INTC            2009       2022
MCD             2009       2022
MSFT            2009       2022
NVDA            2009       2022
PCG             2009       2022
PYPL            2014       2022
SHLDQ           2009       2018



In [4]:
import pandas as pd

# Load and prep
df = pd.read_csv("data/Financial Statements.csv")

# Apply sector mapping (copy from 03-load-financials.py — keep it self-contained)
sector_map = {
    "IT": "Technology",
    "Tech": "Technology",
    "Software": "Technology",
    "FinTech": "FinTech",
    "Bank": "Banking",
    "BANK": "Banking",
    "Finance": "Finance",
    "Manufacturing": "Manufacturing",
    "ELEC": "Electronics",
    "Electronics": "Electronics",
    "LOGI": "Logistics",       # AMZN
    "FOOD": "Food & Beverage",  # MCD
}
df["sector"] = df["Category"].map(sector_map)

# Filter to 2009-2022 analysis window
df = df[(df["Year"] >= 2009) & (df["Year"] <= 2022)]

# Drop any rows with null revenue (shouldn't be many)
df = df.dropna(subset=["Revenue"])

print(f"Working with {len(df)} rows across {df['Company '].nunique()} companies")
print()

Working with 159 rows across 12 companies



<!-- For Each Company: -->

1. Find the company's first reporting year and revenue
2. Find the company's last reporting year (≤2022) and revenue
3. Compute CAGR = (last/first)^(1/years) - 1
4. Sort companies by CAGR